In [19]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from safetensors.torch import load_file
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.model_selection import GroupShuffleSplit

In [15]:
#Paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"
SI_PATH = MODEL_DIR / "semeval-si-classifier"
TC_PATH = MODEL_DIR / "semeval-roberta-classifier"

In [ ]:
#Run training notebooks if models are missing
if not SI_PATH.exists():
    print("SI Model missing. Running training notebook...")
    %run 4.1-fp-semeval-si-modeling.ipynb
if not TC_PATH.exists():
    print("TC Model missing. Running training notebook...")
    %run 4.2-fp-semeval-tc-modeling.ipynb

In [16]:
#Load unified SemEval dataset
df = pd.read_csv(DATA_DIR / "semeval_tc_cleaned.csv")
df.head()

,article_id,text_content,span_text,sentiment,punct_count,lexical_diversity,Appeal_to_Authority,Appeal_to_fear-prejudice,Bandwagon,Black-and-White_Fallacy,...,Loaded_Language,Minimisation,Name_Calling,Red_Herring,Reductio_ad_hitlerum,Repetition,Slogans,Straw_Men,Thought-terminating_Cliches,Whataboutism
0,111111111,Next plague outbreak in Madagascar could be 's...,appeared,0.000000,0,1.000000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,111111111,Next plague outbreak in Madagascar could be 's...,The next transmission could be more pronounced...,0.250000,0,1.000000,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,111111111,Next plague outbreak in Madagascar could be 's...,"a very, very different",0.000000,0,1.000000,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,111111111,Next plague outbreak in Madagascar could be 's...,He also pointed to the presence of the pneumon...,0.483333,0,0.863636,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,111111111,Next plague outbreak in Madagascar could be 's...,but warned that the danger was not over,0.000000,0,1.000000,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
#Constants
FEATURE_COLS = ['sentiment', 'punct_count', 'lexical_diversity']
METADATA_COLS = ['article_id', 'text_content', 'span_text']
LABEL_COLS = [c for c in df.columns if c not in (METADATA_COLS + FEATURE_COLS)]

In [20]:
#Replicate the Test Split (Article-based to prevent leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
_, test_idx = next(gss.split(df, groups=df['article_id']))
test_df = df.iloc[test_idx].copy()

In [21]:
#Load models and tokenizers
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
si_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
si_model = AutoModelForSequenceClassification.from_pretrained(SI_PATH).to(device)

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '../models/semeval-si-classifier'. Use `repo_type` argument if needed.

In [6]:
#Define hybrid class for technique classification
class RoBERTaHybrid(torch.nn.Module):
    def __init__(self, model_name, num_labels, num_extra_features):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.classifier = torch.nn.Linear(768 + num_extra_features, num_labels)
    def forward(self, input_ids, attention_mask, extra_features):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        return SequenceClassifierOutput(logits=self.classifier(torch.cat((out, extra_features), dim=1)))


In [9]:
#Load TC weights (Safetensors)
tc_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
tc_model = RoBERTaHybrid("roberta-base", num_labels=14, num_extra_features=3).to(device)
tc_model.load_state_dict(load_file(TC_PATH / "model.safetensors", device="cpu"))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RuntimeError: Error(s) in loading state_dict for RoBERTaHybrid:
	size mismatch for classifier.weight: copying a param with shape torch.Size([19, 771]) from checkpoint, the shape in current model is torch.Size([14, 771]).
	size mismatch for classifier.bias: copying a param with shape torch.Size([19]) from checkpoint, the shape in current model is torch.Size([14]).

In [10]:
#Define pipeline
def predict_pipeline(text, features):
    #SI Step
    inputs = si_tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    is_prop = torch.argmax(si_model(**inputs).logits, dim=1).item() == 1
    if not is_prop: return "Non-Propaganda", []

    #TC Step
    tc_in = tc_tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    feat = torch.tensor([features], dtype=torch.float).to(device)
    logits = tc_model(tc_in['input_ids'], tc_in['attention_mask'], feat).logits
    probs = torch.sigmoid(logits).cpu().detach().numpy()[0]
    return "Propaganda", [label_cols[i] for i, p in enumerate(probs) if p > 0.6]


In [11]:
test_df['pipeline_res'] = test_df.apply(lambda x: predict_pipeline(x['span_text'], [x['sentiment'], x['punct_count'], x['lexical_diversity']]), axis=1)

NameError: name 'test_df' is not defined

In [12]:
#Categorize results for visualization
def get_status(row):
    actual_is_prop = row[label_cols].sum() > 0
    pred_is_prop = row['pipeline_res'][0] == "Propaganda"
    if not actual_is_prop and not pred_is_prop: return "True Negative (Correct)"
    if not actual_is_prop and pred_is_prop: return "False Positive (SI Hallucination)"
    if actual_is_prop and not pred_is_prop: return "False Negative (SI Missed)"
    return "True Positive (Passed to TC)"

test_df['Status'] = test_df.apply(get_status, axis=1)

NameError: name 'test_df' is not defined

In [ ]:
#Visualize pipeline
plt.figure(figsize=(10, 5))
sns.countplot(data=test_df, y='Status', palette='magma', order=test_df['Status'].value_counts().index)
plt.title("End-to-End Pipeline Performance")
plt.show()

In [13]:
print("\n--- PIPELINE FAILURES (DEBUGGING) ---")
for _, row in test_df[test_df['Status'].str.contains("False")].head(3).iterrows():
    print(f"TEXT: {row['span_text'][:70]}...")
    print(f"ISSUE: {row['Status']} | Actual: {row[label_cols].idxmax() if 'Propaganda' in row['Status'] else 'None'}\n")


--- PIPELINE FAILURES (DEBUGGING) ---


NameError: name 'test_df' is not defined